# Full manual lineage: Semiconductor economic pulse

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://api.bls.gov/publicAPI/v1/timeseries/data/. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "10_semiconductor_supply_chain"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://api.bls.gov/publicAPI/v1/timeseries/data/"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'1d9c6452da0a37f6': '305f095cbcb048283afb5d89cd1579eab90cfcdaa32dc02df45060094d63fe11', '5ad9abca487149bd': '8b3fc861266090f25cd7dce5ea89bf579801c6718230d63bec8e8939707d8c07', 'd403ee8b4b6bcef0': '293d223075a87f24878c0ae883b9c21afe6bbbb16d2d9aee3b25fc9f116c3bd3'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Aligned national indicators with explicit population and revision limitations."""


import numpy as np
import pandas as pd

SERIES = {
    "CES3133440001": ("all_employee_employment", "thousands_of_jobs"),
    "CES3133440007": ("production_nonsupervisory_weekly_hours", "hours_per_week"),
    "PCU334413334413": ("producer_price_index", "index"),
}


def parse_bls(payload: dict, series_id: str) -> pd.DataFrame:
    """Validate response identity and retain monthly observations without M13."""
    if payload.get("status") != "REQUEST_SUCCEEDED":
        raise ValueError(f"BLS request failed for {series_id}: {payload.get('message')}")
    series = payload["Results"]["series"]
    if len(series) != 1 or series[0]["seriesID"] != series_id:
        raise ValueError(f"BLS response identity differs for {series_id}")
    metric, unit = SERIES[series_id]
    rows = [
        {
            "period": f"{item['year']}-{item['period'][1:]}-01",
            "series_id": series_id,
            "metric": metric,
            "value": item["value"],
            "unit": unit,
            "source": "BLS",
        }
        for item in series[0]["data"]
        if item["period"] in {f"M{month:02d}" for month in range(1, 13)}
    ]
    if not rows:
        raise ValueError(f"No monthly observations for {series_id}")
    return pd.DataFrame(rows)


def pulse_tables(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    x = frame.copy()
    x["period"] = pd.to_datetime(x["period"], errors="raise")
    if (
        x.duplicated(["series_id", "period"]).any()
        or x[["series_id", "period", "value"]].isna().any().any()
    ):
        raise ValueError("BLS series/month grain must be unique and observed")
    wide = x.pivot(index="period", columns="series_id", values="value").sort_index()
    if len(wide) < 24:
        raise ValueError("At least 24 monthly observations are required")
    wide = wide.reindex(pd.date_range(wide.index.min(), wide.index.max(), freq="MS"))
    wide.index.name = "period"
    common = wide.dropna()
    if common.empty or common.iloc[0].eq(0).any():
        raise ValueError("A nonzero common rebasing month is required")
    base_date = common.index[0]
    index = 100 * wide / common.iloc[0]
    yoy = 100 * (wide / wide.shift(12) - 1)
    prior_mean = yoy.shift().rolling(12, min_periods=12).mean()
    prior_sd = yoy.shift().rolling(12, min_periods=12).std().replace(0, np.nan)
    z = (yoy - prior_mean) / prior_sd
    long = index.reset_index().melt(
        id_vars="period", var_name="series_id", value_name="rebased_index"
    )
    long = long.merge(
        yoy.reset_index().melt(id_vars="period", var_name="series_id", value_name="yoy_pct"),
        on=["period", "series_id"],
        validate="one_to_one",
    )
    long = long.merge(
        z.reset_index().melt(
            id_vars="period", var_name="series_id", value_name="trailing_momentum_z"
        ),
        on=["period", "series_id"],
        validate="one_to_one",
    )
    long["base_month"] = base_date
    rows = []
    for source in yoy:
        for target in yoy:
            if source == target:
                continue
            for lag in range(7):
                pair = pd.concat(
                    [yoy[source].shift(lag).rename("source"), yoy[target].rename("target")], axis=1
                ).dropna()
                correlation = (
                    pair.source.corr(pair.target)
                    if len(pair) >= 12 and pair.nunique().min() > 1
                    else np.nan
                )
                rows.append(
                    {
                        "source_series": source,
                        "target_series": target,
                        "source_lead_months": lag,
                        "paired_months": len(pair),
                        "descriptive_correlation": correlation,
                    }
                )
    # A shift screen is deliberately not labeled a dated structural-break estimate.
    recent = yoy.iloc[-6:].mean()
    prior = yoy.iloc[-12:-6].mean()
    shifts = pd.DataFrame(
        {
            "series_id": yoy.columns,
            "recent_6m_momentum": recent.values,
            "previous_6m_momentum": prior.values,
            "momentum_shift_pp": (recent - prior).values,
        }
    )
    return {
        "aligned_pulse": long,
        "lead_lag_screen": pd.DataFrame(rows),
        "momentum_shift_screen": shifts,
    }

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def _bls_semiconductors() -> pd.DataFrame:

    tables = []
    for series_id in SERIES:
        response = _get(f"https://api.bls.gov/publicAPI/v1/timeseries/data/{series_id}")
        table = parse_bls(response.json(), series_id)
        preserve_response(
            response,
            project_dir("10_semiconductor_supply_chain") / "data/raw",
            filename=f"{series_id}.json",
            publisher="US Bureau of Labor Statistics",
            source_url="https://www.bls.gov/developers/",
            coverage=f"{table.period.min()} through {table.period.max()}",
            grain="Series and month",
            refresh_frequency="Monthly, subject to revision",
            reuse_basis="Public US government statistical data",
            attribution="US Bureau of Labor Statistics, CES and PPI",
            limitations="Different industry and worker populations; preliminary figures and revisions. Raw response preserves footnotes.",
        )
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


Project: projects/semiconductor_supply_chain
Publisher: https://api.bls.gov/publicAPI/v1/timeseries/data/
Pinned response folder: projects/semiconductor_supply_chain/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
raw = _bls_semiconductors()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 3
Imported rows: 95 columns: 6


,period,series_id,metric,value,unit,source
0,2026-08-01,CES3133440001,all_employee_employment,370.4,thousands_of_jobs,BLS
1,2026-07-01,CES3133440001,all_employee_employment,369.6,thousands_of_jobs,BLS
2,2026-06-01,CES3133440001,all_employee_employment,368.3,thousands_of_jobs,BLS


## 3 Transform and clean

Run the original Semiconductor economic pulse transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def semiconductor_indicators(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(x, ["period", "series_id", "metric", "value", "unit", "source"])
    x["period"] = pd.to_datetime(x["period"], errors="raise")
    x["value"] = pd.to_numeric(x["value"], errors="raise")
    if not x["series_id"].isin(SERIES).all():
        raise ValueError("Unrecognized BLS series; add an explicit population contract first")
    x["metric"] = x["series_id"].map({key: value[0] for key, value in SERIES.items()})
    x["unit"] = x["series_id"].map({key: value[1] for key, value in SERIES.items()})
    aligned = pulse_tables(x)["aligned_pulse"].rename(
        columns={
            "rebased_index": "normalized_index",
            "trailing_momentum_z": "pressure_zscore",
        }
    )
    x = x.merge(aligned, on=["series_id", "period"], validate="one_to_one")
    prior = x[["series_id", "period", "value"]].copy()
    prior["period"] = prior["period"] + pd.DateOffset(months=3)
    prior = prior.rename(columns={"value": "prior_3m_value"})
    x = x.merge(prior, on=["series_id", "period"], how="left", validate="one_to_one")
    x["three_month_change_pct"] = 100 * (x["value"] / x["prior_3m_value"].replace(0, np.nan) - 1)
    x["latest_flag"] = x["period"].eq(x.groupby("series_id")["period"].transform("max")).astype(int)
    x = x.sort_values(["series_id", "period"])
    x["momentum_band"] = np.select(
        [x["three_month_change_pct"] >= 2, x["three_month_change_pct"] <= -2],
        ["Accelerating", "Contracting"],
        default="Stable",
    )
    x["supply_pressure_flag"] = (
        (x["metric"] == "producer_price_index") & (x["yoy_pct"] >= 5)
    ).astype(int)
    return x[
        [
            "period",
            "series_id",
            "metric",
            "value",
            "unit",
            "source",
            "yoy_pct",
            "three_month_change_pct",
            "latest_flag",
            "momentum_band",
            "normalized_index",
            "pressure_zscore",
            "supply_pressure_flag",
        ]
    ]

clean = semiconductor_indicators(raw)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 95 columns: 13


,period,series_id,metric,value,unit,source,yoy_pct,three_month_change_pct,latest_flag,momentum_band,normalized_index,pressure_zscore,supply_pressure_flag
31,2024-01-01,CES3133440001,all_employee_employment,397.4,thousands_of_jobs,BLS,NaN,NaN,0,Stable,100.000000,NaN,0
30,2024-02-01,CES3133440001,all_employee_employment,396.7,thousands_of_jobs,BLS,NaN,NaN,0,Stable,99.823855,NaN,0
29,2024-03-01,CES3133440001,all_employee_employment,394.9,thousands_of_jobs,BLS,NaN,NaN,0,Stable,99.370911,NaN,0


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the divergence_analysis output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def supply_chain_divergence(frame: pd.DataFrame) -> pd.DataFrame:
    pivot = frame.pivot(index="period", columns="metric", values="normalized_index").sort_index()
    if {"production_nonsupervisory_weekly_hours", "all_employee_employment"}.issubset(pivot):
        pivot["hours_employment_divergence"] = (
            pivot["production_nonsupervisory_weekly_hours"] - pivot["all_employee_employment"]
        )
    if {"producer_price_index", "all_employee_employment"}.issubset(pivot):
        pivot["price_employment_divergence"] = (
            pivot["producer_price_index"] - pivot["all_employee_employment"]
        )
    return pivot.reset_index()

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["divergence_analysis"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 3, 'bootstrap_gap': 1, 'feature_diagnostics': 5, 'temporal_diagnostics': 32, 'threshold_sensitivity': 5, 'decision_queue': 20, 'divergence_analysis': 32, 'aligned_pulse': 96, 'lead_lag_screen': 42, 'momentum_shift_screen': 3}


metric,period,all_employee_employment,producer_price_index,production_nonsupervisory_weekly_hours,hours_employment_divergence,price_employment_divergence
0,2024-01-01,100.000000,100.000000,100.000000,0.000000,0.000000
1,2024-02-01,99.823855,99.880149,100.000000,0.176145,0.056294
2,2024-03-01,99.370911,99.757144,99.246231,-0.124680,0.386233
3,2024-04-01,99.194766,100.305936,98.994975,-0.199791,1.111170
4,2024-05-01,99.345747,100.116697,99.748744,0.402996,0.770950


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
KEY_COLUMNS = ["series_id","period"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 95 clean rows: 95
Quality: {'rows': 95, 'columns': 13, 'duplicate_rows': 0, 'missing_cells': 117, 'missing_rate': 0.094737, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after the checks pass. It writes the dated extract, quality report, analytical tables, and source evidence under `data/manual_rebuild/`. If a fresh response has the same date as an existing rebuild, Save uses a source-digest suffix so the verified result is not overwritten; it refuses a conflicting partial folder.

**Check before continuing:** every named output exists in the dated folder; the source digest and reported row counts match the current response.


In [6]:
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    prior_file = OUTPUT_DIR / "quality_report.json"
    prior_hashes = json.loads(prior_file.read_text(encoding="utf-8")).get("source_sha256") if prior_file.is_file() else None
    if isinstance(prior_hashes, str):
        prior_hashes = [prior_hashes]
    if prior_hashes != CURRENT_SOURCE_SHA:
        digest_suffix = hashlib.sha256(json.dumps(CURRENT_SOURCE_SHA).encode("utf-8")).hexdigest()[:12]
        OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / f"{SNAPSHOT_DATE}-{digest_suffix}"
        if OUTPUT_DIR.exists():
            raise FileExistsError(f"A different or partial rebuild already exists at {OUTPUT_DIR}; review it before saving")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "BLS CES and PPI",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


Saved manual rebuild to: projects/semiconductor_supply_chain/data/manual_rebuild/2026-09-26


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `schema.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `populate.sql`, then run `validation.sql` and `analysis.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [7]:
display(analysis_outputs["divergence_analysis"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


metric,period,all_employee_employment,producer_price_index,production_nonsupervisory_weekly_hours,hours_employment_divergence,price_employment_divergence
0,2024-01-01,100.000000,100.000000,100.000000,0.000000,0.000000
1,2024-02-01,99.823855,99.880149,100.000000,0.176145,0.056294
2,2024-03-01,99.370911,99.757144,99.246231,-0.124680,0.386233
3,2024-04-01,99.194766,100.305936,98.994975,-0.199791,1.111170
4,2024-05-01,99.345747,100.116697,99.748744,0.402996,0.770950
5,2024-06-01,99.396074,99.407052,98.994975,-0.401100,0.010978
6,2024-07-01,99.421238,96.518009,98.241206,-1.180032,-2.903229
7,2024-08-01,99.345747,96.057529,98.994975,-0.350772,-3.288219
8,2024-09-01,99.245093,96.155302,98.994975,-0.250118,-3.089791
9,2024-10-01,98.993457,96.902794,98.492462,-0.500995,-2.090663


Tableau input: projects/semiconductor_supply_chain/data/manual_rebuild/2026-09-26/tableau_extract.csv


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [9]:
DECISION_TEXT = (
    "For the pinned 2026-09-26 BLS snapshot, review the August 2026 "
    "price-employment divergence: the rebased producer-price index is 89.80 "
    "versus 93.21 for employment, a -3.41-point gap. Weekly-hours data end "
    "in July, so do not treat the missing August hours as a current signal. "
    "This is descriptive evidence for follow-up, not proof of a supply squeeze."
)
INTERPRETATION_LIMIT = (
    "These series are not direct inventory or capacity measures. Publication "
    "lags, changing series definitions, revisions, and base-period choices "
    "limit comparison; lead-lag correlations do not establish causality."
)
EVIDENCE_FILES = [
    "quality_report.json",
    "analysis/divergence_analysis.csv",
]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)

Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/divergence_analysis.csv']
Decision: For the pinned 2026-09-26 BLS snapshot, review the August 2026 price-employment divergence: the rebased producer-price index is 89.80 versus 93.21 for employment, a -3.41-point gap. Weekly-hours data end in July, so do not treat the missing August hours as a current signal. This is descriptive evidence for follow-up, not proof of a supply squeeze.
Limit: These series are not direct inventory or capacity measures. Publication lags, changing series definitions, revisions, and base-period choices limit comparison; lead-lag correlations do not establish causality.
